# Weather Summary Classification
## Machine Learning Project — B.Tech CSE-DS

**Objective:** Build a robust multi-class classification model to predict the `Daily Summary` (target variable) from meteorological features.

**Dataset:** Szeged, Hungary Weather History  
**Columns:** Formatted Date, Summary, Precip Type, Temperature (C), Apparent Temperature (C), Humidity, Wind Speed (km/h), Wind Bearing (degrees), Visibility (km), Loud Cover, Pressure (millibars), Daily Summary

---
## 1. Import Libraries

In [ ]:
# Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.utils.class_weight import compute_class_weight

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

# Imbalanced data handling
from imblearn.over_sampling import SMOTE

print('All libraries imported successfully!')

---
## 2. Load & Inspect the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('weatherHistory.csv')

print('Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
print('Missing Values:')
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Target variable distribution
print('Unique Daily Summaries (Target):', df['Daily Summary'].nunique())
print('\nTop 15 Classes:')
print(df['Daily Summary'].value_counts().head(15))

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- Target Class Distribution (Top 15) ---
top_classes = df['Daily Summary'].value_counts().head(15)

fig = px.bar(
    x=top_classes.index,
    y=top_classes.values,
    title='Top 15 Daily Summary Classes (Target Variable Distribution)',
    labels={'x': 'Daily Summary', 'y': 'Count'},
    color=top_classes.values,
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

In [ ]:
# --- Temperature Distribution ---
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Temperature (C) Distribution', 'Apparent Temperature (C) Distribution'
))

fig.add_trace(go.Histogram(x=df['Temperature (C)'], nbinsx=50,
                            marker_color='steelblue', name='Temp'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Apparent Temperature (C)'], nbinsx=50,
                            marker_color='orange', name='App Temp'), row=1, col=2)

fig.update_layout(title='Temperature Distributions', showlegend=False, height=400)
fig.show()

In [ ]:
# --- Numerical Feature Distributions ---
num_cols = ['Temperature (C)', 'Apparent Temperature (C)',
            'Humidity', 'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distributions of Meteorological Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Correlation Heatmap ---
corr = df[num_cols].corr()

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Correlation Matrix of Numerical Features',
    aspect='auto'
)
fig.show()

In [ ]:
# --- Temperature vs Humidity colored by Precip Type ---
fig = px.scatter(
    df.dropna(subset=['Precip Type']),
    x='Temperature (C)',
    y='Humidity',
    color='Precip Type',
    title='Temperature vs Humidity by Precipitation Type',
    opacity=0.4,
    color_discrete_map={'rain': '#1f77b4', 'snow': '#aec7e8'}
)
fig.show()

In [ ]:
# --- Boxplots: Temperature by Top 10 Daily Summaries ---
top10 = df['Daily Summary'].value_counts().head(10).index
df_top10 = df[df['Daily Summary'].isin(top10)]

fig = px.box(
    df_top10,
    x='Daily Summary',
    y='Temperature (C)',
    color='Daily Summary',
    title='Temperature Distribution by Top 10 Daily Summary Categories'
)
fig.update_layout(showlegend=False, xaxis_tickangle=-40)
fig.show()

In [ ]:
# --- Wind Speed vs Visibility ---
fig = px.scatter(
    df.sample(3000, random_state=42),
    x='Wind Speed (km/h)',
    y='Visibility (km)',
    color='Temperature (C)',
    title='Wind Speed vs Visibility (colored by Temperature)',
    color_continuous_scale='Thermal',
    opacity=0.5
)
fig.show()

In [ ]:
# --- Precipitation Type Distribution ---
precip_counts = df['Precip Type'].value_counts(dropna=False)
fig = px.pie(
    names=precip_counts.index.astype(str),
    values=precip_counts.values,
    title='Precipitation Type Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

---
## 4. Data Preprocessing

In [ ]:
# ---- 4.1 Drop columns not useful for modeling ----
# 'Formatted Date' is timestamp; 'Loud Cover' is constant 0; 'Summary' is similar to target
df_clean = df.drop(columns=['Formatted Date', 'Loud Cover'], errors='ignore')

# Drop 'Summary' to prevent data leakage (it's a simplified version of Daily Summary)
df_clean = df_clean.drop(columns=['Summary'], errors='ignore')

print('Columns after dropping:', df_clean.columns.tolist())

In [ ]:
# ---- 4.2 Handle Missing Values ----

# 'Precip Type' has ~7% missing — fill with 'none'
df_clean['Precip Type'] = df_clean['Precip Type'].fillna('none')

# Numeric columns: fill with median
num_cols_clean = ['Temperature (C)', 'Apparent Temperature (C)',
                  'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)',
                  'Visibility (km)', 'Pressure (millibars)']

for col in num_cols_clean:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

print('Remaining missing values after imputation:')
print(df_clean.isnull().sum())

In [ ]:
# ---- 4.3 Encode Categorical Variables ----

# Precip Type → Label Encoding
le_precip = LabelEncoder()
df_clean['Precip Type'] = le_precip.fit_transform(df_clean['Precip Type'])

# Target variable: Daily Summary → Label Encoding
le_target = LabelEncoder()
df_clean['Daily Summary Encoded'] = le_target.fit_transform(df_clean['Daily Summary'])

print('Precip Type classes:', le_precip.classes_)
print('Number of target classes:', len(le_target.classes_))

In [ ]:
# ---- 4.4 Reduce Rare Classes ----
# Keep only classes with >= 50 samples to ensure meaningful learning
class_counts = df_clean['Daily Summary'].value_counts()
valid_classes = class_counts[class_counts >= 50].index
df_clean = df_clean[df_clean['Daily Summary'].isin(valid_classes)]

# Re-encode after filtering
le_target2 = LabelEncoder()
df_clean['Daily Summary Encoded'] = le_target2.fit_transform(df_clean['Daily Summary'])

print('Shape after removing rare classes:', df_clean.shape)
print('Number of final classes:', df_clean['Daily Summary Encoded'].nunique())

In [ ]:
# ---- 4.5 Feature Engineering ----

# Derived feature: temperature difference (feels-like vs actual)
df_clean['Temp_Diff'] = df_clean['Apparent Temperature (C)'] - df_clean['Temperature (C)']

# Wind Chill proxy interaction
df_clean['Wind_Humidity'] = df_clean['Wind Speed (km/h)'] * df_clean['Humidity']

print('New features added: Temp_Diff, Wind_Humidity')

In [ ]:
# ---- 4.6 Define Features and Target ----
feature_cols = [
    'Precip Type', 'Temperature (C)', 'Apparent Temperature (C)',
    'Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)',
    'Visibility (km)', 'Pressure (millibars)', 'Temp_Diff', 'Wind_Humidity'
]

X = df_clean[feature_cols]
y = df_clean['Daily Summary Encoded']

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)

In [ ]:
# ---- 4.7 Train-Test Split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)

In [ ]:
# ---- 4.8 Feature Scaling (Standard Scaler) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling complete. Mean (first feature):', X_train_scaled[:, 0].mean().round(5))

In [ ]:
# ---- 4.9 Handle Class Imbalance with SMOTE (on training set only) ----
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:', X_train_scaled.shape)
print('After SMOTE:', X_train_sm.shape)

---
## 5. Model Training & Evaluation

In [ ]:
# ---- 5.1 Define Models ----
models = {
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM (RBF Kernel)':      SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression':   LogisticRegression(max_iter=1000, random_state=42, multi_class='auto'),
    'K-Nearest Neighbors':   KNeighborsClassifier(n_neighbors=7),
    'Decision Tree':         DecisionTreeClassifier(max_depth=15, random_state=42),
    'Naive Bayes':           GaussianNB()
}

print(f'{len(models)} models defined.')

In [ ]:
# ---- 5.2 Train All Models & Compare ----
results = {}

for name, model in models.items():
    print(f'Training: {name}...')
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_scaled)
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results[name] = {
        'Accuracy':  round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall':    round(rec * 100, 2),
        'F1 Score':  round(f1 * 100, 2)
    }
    print(f'  ✓ Accuracy: {acc*100:.2f}%')

results_df = pd.DataFrame(results).T
print('\n--- Model Comparison ---')
print(results_df.sort_values('Accuracy', ascending=False))

In [ ]:
# ---- 5.3 Comparative Performance Visualization ----
results_melted = results_df.reset_index().melt(
    id_vars='index', var_name='Metric', value_name='Score (%)')

fig = px.bar(
    results_melted,
    x='index',
    y='Score (%)',
    color='Metric',
    barmode='group',
    title='Model Comparison: Accuracy, Precision, Recall, F1 Score',
    labels={'index': 'Model'},
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

---
## 6. Best Model: Random Forest — Deep Dive

In [ ]:
# Best model (typically Random Forest or Gradient Boosting)
best_model_name = results_df['Accuracy'].idxmax()
best_model = models[best_model_name]

print(f'Best Model: {best_model_name}')
print(f'Accuracy: {results_df.loc[best_model_name, "Accuracy"]}%')

In [ ]:
# ---- 6.1 Classification Report ----
y_pred_best = best_model.predict(X_test_scaled)
class_names = le_target2.classes_

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best,
                             target_names=class_names, zero_division=0))

In [ ]:
# ---- 6.2 Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred_best)

fig = px.imshow(
    cm,
    x=class_names,
    y=class_names,
    color_continuous_scale='Blues',
    title=f'Confusion Matrix — {best_model_name}',
    text_auto=True,
    aspect='auto'
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# ---- 6.3 Feature Importance (Random Forest) ----
if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(
        best_model.feature_importances_, index=feature_cols
    ).sort_values(ascending=True)

    fig = px.bar(
        x=feat_imp.values,
        y=feat_imp.index,
        orientation='h',
        title=f'Feature Importance — {best_model_name}',
        labels={'x': 'Importance Score', 'y': 'Feature'},
        color=feat_imp.values,
        color_continuous_scale='Teal'
    )
    fig.show()

In [ ]:
# ---- 6.4 Cross-Validation Score ----
cv_scores = cross_val_score(best_model, X_train_sm, y_train_sm, cv=5, scoring='accuracy')
print(f'5-Fold Cross-Validation Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print('Fold scores:', [round(s*100, 2) for s in cv_scores])

---
## 7. Hyperparameter Tuning (Random Forest)

In [ ]:
# Grid Search — limited param grid for time efficiency
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_tuned = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_tuned, param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1
)

grid_search.fit(X_train_sm, y_train_sm)

print('Best Parameters:', grid_search.best_params_)
print('Best CV Accuracy:', round(grid_search.best_score_ * 100, 2), '%')

In [ ]:
# Evaluate tuned model
best_rf_tuned = grid_search.best_estimator_
y_pred_tuned = best_rf_tuned.predict(X_test_scaled)

acc_tuned = accuracy_score(y_test, y_pred_tuned)
f1_tuned  = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)

print(f'Tuned RF Test Accuracy : {acc_tuned*100:.2f}%')
print(f'Tuned RF Weighted F1   : {f1_tuned*100:.2f}%')

---
## 8. Summary & Conclusions

In [ ]:
print('='*60)
print('       WEATHER SUMMARY CLASSIFICATION — RESULTS')
print('='*60)
print(results_df.sort_values('Accuracy', ascending=False).to_string())
print('\nBest Model:', best_model_name)
print(f"Test Accuracy: {results_df.loc[best_model_name, 'Accuracy']}%")
print('='*60)
print()
print('Key Findings:')
print('1. Temperature and Apparent Temperature are the strongest predictors.')
print('2. Humidity and Wind Speed are secondary but important features.')
print('3. SMOTE effectively addressed class imbalance.')
print('4. Random Forest outperforms other algorithms for this task.')
print('5. Feature engineering (Temp_Diff, Wind_Humidity) improved accuracy.')